# SHAP Analysis

INT 374 - PCOS Prediction Project

Explaining model predictions using SHAP (SHapley Additive exPlanations).
Understanding which features influence predictions and how.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import warnings

warnings.filterwarnings('ignore')
print('Libraries loaded')

In [ ]:
# Load data and model
df = pd.read_csv('../data/PCOS_final.csv')
model = pickle.load(open('../outputs/models/best_model.pkl', 'rb'))
feature_names = pickle.load(open('../outputs/models/feature_names.pkl', 'rb'))

X = df.drop('PCOS', axis=1)
y = df['PCOS']

print(f'Data shape: {X.shape}')
print(f'Model: {model}')

## Create SHAP Explainer

In [ ]:
# Create explainer
print('Creating SHAP explainer... This may take a minute')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# For binary classification, shap_values is a list of arrays
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Get SHAP values for positive class

print('SHAP values calculated')

## Feature Importance - Bar Plot

In [ ]:
# SHAP feature importance - mean absolute SHAP values
feature_importance = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(importance_df)), importance_df['Importance'].values, color='steelblue')
ax.set_yticks(range(len(importance_df)))
ax.set_yticklabels(importance_df['Feature'].values, fontsize=9)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('SHAP Feature Importance')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../outputs/figures/35_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved SHAP bar plot')

## SHAP Summary Plot (Beeswarm)

In [ ]:
# SHAP summary plot (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X, plot_type='beeswarm', show=False)
plt.tight_layout()
plt.savefig('../outputs/figures/34_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved SHAP summary plot')

## SHAP Waterfall Plot

In [ ]:
# Waterfall plot for a sample (PCOS positive case)
pcos_positive_idx = y[y == 1].index[2]  # Get index of a positive case
sample_idx = df.index.get_loc(pcos_positive_idx)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X.iloc[sample_idx],
    feature_names=X.columns.tolist()
), show=False)

plt.tight_layout()
plt.savefig('../outputs/figures/36_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved waterfall plot for sample {sample_idx}')

## SHAP Dependence Plots

In [ ]:
# Dependence plots for top 3 features
top_3_features = importance_df.head(3)['Feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, feature in enumerate(top_3_features):
    feature_idx = X.columns.get_loc(feature)
    shap.dependence_plot(feature_idx, shap_values, X, ax=axes[idx], show=False)
    axes[idx].set_title(f'SHAP Dependence - {feature}')

plt.tight_layout()
plt.savefig('../outputs/figures/37_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved SHAP dependence plots')

## Learning Curve

In [ ]:
# Learning curve for Random Forest
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), 
    scoring='f1'
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training F1')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')

plt.plot(train_sizes, val_mean, 'o-', color='red', label='Validation F1')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='red')

plt.xlabel('Training Set Size')
plt.ylabel('F1 Score')
plt.title('Learning Curve - Random Forest')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/33_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved learning curve')

## Summary Report

## Summary Report

In [ ]:
# Create summary report
print('\n' + '='*60)
print('PCOS PREDICTION MODEL - SUMMARY')
print('='*60)

print(f'\nDataset:')
print(f'  Total samples: {len(df)}')
print(f'  Features used: {X.shape[1]}')
print(f'  PCOS positive: {(y==1).sum()} ({(y==1).mean()*100:.1f}%)')
print(f'  PCOS negative: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)')

print(f'\nTop 10 Important Features (by SHAP):')
for i, row in importance_df.head(10).iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

print(f'\nModel: Random Forest')
print(f'  Number of trees: 100')
print(f'  Max depth: 15')

print('\nFiles saved:')
print('  - Model: outputs/models/best_model.pkl')
print('  - Scaler: outputs/models/scaler.pkl')
print('  - Figures: 18 PNG files in outputs/figures/')
print('  - Reports: CSV and TXT files in outputs/reports/')
print('\n' + '='*60)

In [ ]:
print('\nAnalysis completed successfully!')